# jax-dpo-min on Colab TPU

Runs the Gemma-2B-IT recipe on a Colab TPU v5e-1 or v6e-1. bf16 for throughput. Single-device — no pmap, no sharding (multi-device is an explicit non-goal per the repo's design).

**Runtime type: TPU.** Pick this from Runtime → Change runtime type before running any cells.

**Important**: training and eval run **in-kernel** (not via `!python ...`). TPUs only allow one process to own them at a time, so if the kernel claims the TPU and then you shell out to a subprocess, the subprocess will crash with `ABORTED: The TPU is already in use`. The in-kernel pattern avoids that entirely.

## 1. Install JAX with TPU backend.

Do **not** `import jax` in this cell. We want the kernel to stay uninitialized until we're ready to run the model; otherwise the TPU gets claimed early for nothing.

In [ ]:
!pip install -q -U "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
!pip install -q -U flax optax orbax-checkpoint "transformers>=4.44,<5" datasets sentencepiece pyyaml

## 2. Mount Drive for checkpoint persistence.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone the repo.

In [ ]:
%cd /content
![ -d jax-dpo-min ] || git clone https://github.com/jman4162/jax-dpo-min.git
%cd jax-dpo-min

## 4. HuggingFace auth for Gemma (gated model).

In [ ]:
from huggingface_hub import login
login()  # paste token from https://huggingface.co/settings/tokens when prompted

## 5. Confirm TPU, then train.

This is the first cell that imports `jax` — doing so claims the TPU for this kernel. The `train()` call below runs in the same kernel, so there's no subprocess contention.

Expected wall time: ~30-60 min for 1500 steps on v5e-1.

In [ ]:
import jax
print('devices:', jax.devices())
assert jax.devices()[0].platform == 'tpu', 'No TPU detected — set Runtime → Change runtime type → TPU.'

from train import load_config, train

OUTPUT_DIR = '/content/drive/MyDrive/jax-dpo-min/runs/gemma_tpu_run01'
cfg = load_config('configs/gemma_tpu.yaml')
train(cfg, OUTPUT_DIR)

## 6. Evaluate the final checkpoint.

In [ ]:
import pickle
from pathlib import Path

import jax
import jax.numpy as jnp
from datasets import load_dataset

from eval import pairwise_accuracy
from model import load_model_and_tokenizer

CKPT = Path(OUTPUT_DIR) / f"step_{cfg.num_train_steps:06d}"
with open(CKPT / 'state.pkl', 'rb') as f:
    state = pickle.load(f)
ckpt_cfg = state['config']
lora_params = jax.tree_util.tree_map(jnp.asarray, state['lora_params'])

model, base_params, tokenizer = load_model_and_tokenizer(ckpt_cfg['model'], dtype=ckpt_cfg.get('dtype', 'float32'))
ds = load_dataset(ckpt_cfg['dataset'], split='train[-256:]')
acc = pairwise_accuracy(model, base_params, lora_params, tokenizer, ds, ckpt_cfg)
print(f'pairwise_accuracy: {acc:.4f}  (n={len(ds)})')